# PPO

Notebook with a demonstration PPO implementation, running on several gymnasium environments

In [ ]:
import numpy as np
import torch
import torch.optim as optim
import gymnasium as gym
import matplotlib.pyplot as plt
import os
import sys
from typing import List, Callable, Dict, Optional

from importlib import reload
sys.path.append(os.getcwd())
import PPO

In [ ]:
plt.style.use('dark_background')

device = torch.device("cpu")

#reload(PPO)

In [ ]:
ENV_NAME = "LunarLander-v2"

In [ ]:
ppoTrainerConfig = PPO.PPOTrainerConfig()
ppoTrainerConfig.num_envs = 4
ppoTrainerConfig.async_envs = True
trainer = PPO.PPOTrainer(ENV_NAME, ppoTrainerConfig)

trainConfig = PPO.PPOTrainerTrainConfig()
trainConfig.total_steps = 100_000
ppoConfig = PPO.PPOTrainerPPOUpdateConfig()
run = trainer.train(trainConfig, ppoConfig)['returns']

In [ ]:
def plot_returns(runs: Dict[str, List[float]], smooth: int = 20, title: str = ""):
    fig, ax = plt.subplots(figsize=(9, 4))
    for label, returns in runs.items():
        if len(returns) < smooth:
            ax.plot(returns, alpha=0.6, label=label)
            continue
        r = np.asarray(returns)
        smoothed = np.convolve(r, np.ones(smooth) / smooth, mode='valid')
        ax.plot(np.arange(smooth - 1, len(r)), smoothed, label=label)
    ax.set_xlabel("Episode")
    ax.set_ylabel(f"Return (smoothed over {smooth})")
    ax.set_title(title)
    ax.legend()
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()


plot_returns({"Run" : run})

In [ ]:
def make_env(name: str, render: bool) -> gym.Env:
    render_mode = "rgb_array" if render else None
    env = gym.make(name, render_mode=render_mode)
    return env

In [ ]:
env = make_env(ENV_NAME, render=True)